In [10]:
%cd ..

/media/kirti/Dev/GenAI/TransformerFromScratch


## Data Creation

In [11]:
data_path = "data/kirti.txt"
with open(data_path,"r") as file:
    data = file.read()

len(data)

2237

In [12]:
unique_character = sorted(list(set(data)))

In [13]:
vocab_size = len(unique_character)

In [14]:
int_to_string = {k:v for k,v in enumerate(unique_character)}
string_to_int = {v:k for k,v in enumerate(unique_character)}

In [15]:
string_to_int

{'\n': 0,
 ' ': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '-': 6,
 '.': 7,
 '/': 8,
 ':': 9,
 '?': 10,
 'A': 11,
 'C': 12,
 'D': 13,
 'E': 14,
 'G': 15,
 'I': 16,
 'K': 17,
 'L': 18,
 'M': 19,
 'N': 20,
 'P': 21,
 'Q': 22,
 'T': 23,
 'W': 24,
 'Y': 25,
 'a': 26,
 'b': 27,
 'c': 28,
 'd': 29,
 'e': 30,
 'f': 31,
 'g': 32,
 'h': 33,
 'i': 34,
 'j': 35,
 'k': 36,
 'l': 37,
 'm': 38,
 'n': 39,
 'o': 40,
 'p': 41,
 'r': 42,
 's': 43,
 't': 44,
 'u': 45,
 'v': 46,
 'w': 47,
 'y': 48,
 'z': 49}

In [16]:
def encode(s):
    ans = []
    for char in s:
        ans.append(string_to_int[char])
    return ans

def decode(l):
    ans = ""
    for token in l:
        ans+=int_to_string[token]
    return ans


In [17]:
encode("Kirti")

[17, 34, 42, 44, 34]

In [18]:
decode([17, 34, 42, 44, 34])

'Kirti'

In [19]:
import torch


In [20]:
encoded_data = torch.tensor(encode(data),dtype=torch.long)

In [21]:
len(encoded_data)*90//100

2013

In [22]:
encoded_data.dtype

torch.int64

In [23]:
split_idx = data.rfind("\n\n", 0, int(len(data) * 0.9))

In [24]:
train_data = encoded_data[:split_idx]
val_data = encoded_data[split_idx:]

In [25]:
chunk = train_data[:9]



In [28]:
x = chunk[:-1]
y = chunk[1:]

In [29]:
x,y

(tensor([17, 34, 42, 44, 34,  1, 21, 40]),
 tensor([34, 42, 44, 34,  1, 21, 40, 32]))

In [30]:
torch.randint(0,len(train_data)-8,(4,1))

tensor([[ 250],
        [1974],
        [1043],
        [1458]])

In [31]:
batch_size = 4
block_size = 8
ix = torch.randint(0,len(train_data)-block_size,(batch_size,))

In [32]:
x = torch.stack([train_data[i:i+block_size] for i in ix])
y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])

In [33]:
x,y

(tensor([[42, 44, 34,  1, 21, 40, 32, 42],
         [29, 34, 39, 32,  1, 42, 34, 32],
         [26, 39, 44, 43,  1, 44, 40,  1],
         [46, 30,  1, 23, 42, 26, 39, 43]]),
 tensor([[44, 34,  1, 21, 40, 32, 42, 26],
         [34, 39, 32,  1, 42, 34, 32, 33],
         [39, 44, 43,  1, 44, 40,  1, 27],
         [30,  1, 23, 42, 26, 39, 43, 31]]))

In [34]:
x.shape,y.shape

(torch.Size([4, 8]), torch.Size([4, 8]))

In [35]:
def get_batch(split_type):
    batch_size = 4
    block_size = 8
    data = train_data if split_type=="train" else val_data
    ix = torch.randint(0,len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
                

In [36]:
get_batch("train")

(tensor([[39, 28, 30,  1, 26, 39, 29,  1],
         [ 1, 21, 40, 32, 42, 26,  1, 34],
         [ 1, 20, 30, 30, 38, 45, 28, 33],
         [26, 29, 30, 43, 33,  5,  1, 16]]),
 tensor([[28, 30,  1, 26, 39, 29,  1, 19],
         [21, 40, 32, 42, 26,  1, 34, 43],
         [20, 30, 30, 38, 45, 28, 33,  1],
         [29, 30, 43, 33,  5,  1, 16, 39]]))

In [37]:
get_batch("val")

(tensor([[44, 34, 31, 34, 28, 34, 26, 37],
         [26, 29, 30, 43, 33,  5,  1, 26],
         [21, 42, 26, 29, 30, 43, 33,  5],
         [ 1, 41, 33, 48, 43, 34, 28, 43]]),
 tensor([[34, 31, 34, 28, 34, 26, 37,  1],
         [29, 30, 43, 33,  5,  1, 26, 39],
         [42, 26, 29, 30, 43, 33,  5,  1],
         [41, 33, 48, 43, 34, 28, 43,  5]]))

## Training

In [8]:
from torch import nn
from torch.nn import functional

In [1]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size:int
    d_model:int
    max_seq_len: int # context length
    num_heads:int
    num_layers:int
    dropout:float

In [2]:
model_config = ModelConfig(
    vocab_size=50,
    d_model=32,
    max_seq_len=15,
    num_heads=4,
    num_layers=3,
    dropout=0.2
)

In [4]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        """
        Always remember that embedding layer is not actually a neural laer with any activation function ,
        its just an 2d matrix lookup the matrix size is (vocab_size,d_model) but but but the d_model here is lets one array
         so that array is already have learnable parameter means in our case that 32 dim arrays have the values from the training 
         for e.g:
         token_embeding[1] will be the embeding of token id 1 which is 32 dim embedding array
        """
        self.token_embeddings=nn.Embedding(model_config.vocab_size,model_config.d_model)
        self.pos_embeddings = nn.Embedding(model_config.max_seq_len,model_config.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1],device=token_ids.device)
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        # initialize q,k,v
        self.w_q = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_k = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_v = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

        self.register_buffer(
                            "casual_mask", 
                            torch.tril(
                                torch.ones(
                                    self.config.max_seq_len, 
                                    self.config.max_seq_len
                                    )
                                )
                        )
        
        self.head_dim = self.config.d_model//self.config.num_heads
        

        self.w_o = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

    def forward(self,x):
        # here x is the embeding and not raw x  and b is batch size, t is the current sequence length and d_model is hidden dimension
        # shape of x -> (b,t,d_model)
        q = self.w_q(x) # (b,t,d_model)
        k = self.w_k(x) # (b,t,d_model)
        v = self.w_v(x) # (b,t,d_model)

        q = torch.reshape(q,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        k = torch.reshape(k,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        v = torch.reshape(v,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)


        q = q.transpose(1,2) # (b,num_head,t,head_dim)
        k = k.transpose(1,2) # (b,num_head,t,head_dim)
        v = v.transpose(1,2) # (b,num_head,t,head_dim)

        scores = q @ k.transpose(-2,-1) / (self.head_dim ** 0.5) # (b,num_head,t,t)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]] # (t,t)
        masked_score = scores.masked_fill(mask==0,float("-inf")) # (b,num_head,t,t)
        attention_weights = torch.softmax(masked_score,dim = -1) # (b,num_head,t,t)
        context_vector = attention_weights @ v # (b,num_head,t,head_dim)

        context_vector = context_vector.transpose(1,2) # (b,t,num_head,head_dim) 
        context_vector = context_vector.contiguous().view(x.shape) # (b,t,d_model)
        return self.w_o(context_vector) #(b,t,d_model) which is same as the input embedding 


In [6]:
class FeedForward(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.model = nn.Sequential(
            nn.Linear(self.config.d_model,4*self.config.d_model),
            nn.GELU(),
            nn.Linear(4*self.config.d_model,self.config.d_model)
        )

    def forward(self,x):
        return self.model(x)

In [7]:
class DecoderBlock(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()

        self.config = model_config
        self.attention_layer = MultiHeadAttention(model_config)
        self.ffn = FeedForward(model_config)
        self.ln_1 =  nn.LayerNorm(model_config.d_model) # before attention layer
        self.ln_2 =  nn.LayerNorm(model_config.d_model) # before feed forward layer

    def forward(self,x):
        x =  x+self.attention_layer(self.ln_1(x))
        x = x+ self.ffn(self.ln_2(x))
        return x



In [39]:
class Transformer(nn.Module):
    def __init__(self,model_config:ModelConfig):
        super().__init__()
        self.config = model_config
        self.embedding_layer = EmbeddingLayer(model_config)
        self.drop = nn.Dropout(model_config.dropout)
        self.block = nn.ModuleList([DecoderBlock(model_config) for _ in range(model_config.num_layers)])
        self.ln_f = nn.LayerNorm(model_config.d_model)
        self.lm_head = nn.Linear(model_config.d_model,model_config.vocab_size,bias=False)
        self.lm_head.weight = self.embedding_layer.token_embeddings.weight

        self.apply(self._init_weights)

    def _init_weights(self,module):
        if isinstance(module, nn.Linear):
            module.weight = torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                module.bias = torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            module.weight = nn.init.normal_(module.weight,mean=0,std=0.02)

        
    def forward(self,x,targets=None):
        embedding = self.embedding_layer(x)
        out = self.drop(embedding)
        for i in range(self.config.num_layers):
            out = self.block[i](out)

        out_norm = self.ln_f(out)
        logits = self.lm_head(out_norm)
        loss = None
        if targets is not None:
            loss = functional.cross_entropy(
                logits.view(-1,self.config.vocab_size),
                targets.view(-1))
        return logits,loss

    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:,-self.config.max_seq_len:] #index conditional
            logits,_ = self(idx_cond)
            last_token = logits[:,-1,:]
            probs = functional.softmax(last_token,dim=-1)
            curr_id = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, curr_id), dim=1)

        return idx

In [46]:
def get_batch(split_type,batch_size,block_size):
    data = train_data if split_type=="train" else val_data
    ix = torch.randint(0,len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
                

In [73]:
model = Transformer(model_config)

In [74]:
model

Transformer(
  (embedding_layer): EmbeddingLayer(
    (token_embeddings): Embedding(50, 32)
    (pos_embeddings): Embedding(15, 32)
  )
  (drop): Dropout(p=0.2, inplace=False)
  (block): ModuleList(
    (0-2): 3 x DecoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=32, out_features=32, bias=False)
        (w_k): Linear(in_features=32, out_features=32, bias=False)
        (w_v): Linear(in_features=32, out_features=32, bias=False)
        (w_o): Linear(in_features=32, out_features=32, bias=False)
      )
      (ffn): FeedForward(
        (model): Sequential(
          (0): Linear(in_features=32, out_features=128, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=128, out_features=32, bias=True)
        )
      )
      (ln_1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
      (ln_2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (ln_f): LayerNorm((32,), eps=1e-05, 

In [42]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 39,872


In [47]:
xb,yb = get_batch("train",4,model_config.max_seq_len)

In [48]:
logits,loss = model(xb,yb)

In [49]:
loss.item(),logits.shape

(3.895784854888916, torch.Size([4, 15, 50]))

In [53]:
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

In [55]:
optimizer.zero_grad(set_to_none=True)

In [56]:
loss.backward()

In [57]:
optimizer.step()

In [58]:
_,new_loss = model(xb,yb)
print("Before:", loss.item(), "After:", new_loss.item())

Before: 3.895784854888916 After: 3.756951093673706


In [75]:
@torch.no_grad()
def calculate_loss(eval_iters=20,batch_size=4):
    out = {}
    model.eval()
    train_loss,val_loss=[],[]
    for _ in range(eval_iters):
        x_train,y_train = get_batch("train",batch_size,model_config.max_seq_len)
        _,loss = model(x_train,y_train)
        train_loss.append(loss.item())

        x_val,y_val = get_batch("val",batch_size,model_config.max_seq_len)
        _,loss = model(x_val,y_val)
        val_loss.append(loss.item())

    out = {"train_loss": sum(train_loss)/eval_iters,
           "val_loss":sum(val_loss)/eval_iters}
    model.train()
    return out


In [76]:
max_iters = 2500    # total training steps
eval_interval = 250 # evaluate every 50 steps
batch_size = 8
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3)

In [77]:
for iter in range(max_iters):
    xb,yb = get_batch("train",batch_size,model_config.max_seq_len)
    _,loss = model(xb,yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if iter % eval_interval == 0:
        out = calculate_loss(batch_size=batch_size)
        print(out)

{'train_loss': 3.8484050393104554, 'val_loss': 3.848555064201355}
{'train_loss': 2.3531377792358397, 'val_loss': 2.4843976855278016}
{'train_loss': 1.9902939677238465, 'val_loss': 2.1888253033161162}
{'train_loss': 1.7200723171234131, 'val_loss': 1.9654865562915802}
{'train_loss': 1.560590571165085, 'val_loss': 1.8360981464385986}
{'train_loss': 1.3372968018054963, 'val_loss': 1.6472338140010834}
{'train_loss': 1.2838812232017518, 'val_loss': 1.6976181983947753}
{'train_loss': 1.049462580680847, 'val_loss': 1.5654048919677734}
{'train_loss': 1.012617689371109, 'val_loss': 1.6659859538078308}
{'train_loss': 0.8951471328735352, 'val_loss': 1.4975847482681275}


In [78]:
prompt = torch.tensor([encode("Q: Who ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60)

In [79]:
decode(out[0].tolist())

'Q: Who s Kirti lento cume docume?\nA: Kirti Pogua Pora is an e AI an'

In [80]:
prompt = torch.tensor([encode("Q: Where ")], dtype=torch.long)

In [81]:
out = model.generate(prompt, max_new_tokens=60)
decode(out[0].tolist())

'Q: Where in ML Enjoyl Eng.\n\n\nQ: Wheis Kirti he do w?\nA: Kirti is of w'

In [82]:
prompt = torch.tensor([encode("Kirti is ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60)
decode(out[0].tolist())

'Kirti is lo?\nA: Kirti toA: grave huspace, aceme, shAI/Madelinins, Mac'

In [83]:
prompt = torch.tensor([encode("Q: What degree ")], dtype=torch.long)
out = model.generate(prompt, max_new_tokens=60)
decode(out[0].tolist())

'Q: What degree w?\nA: wout int orllove, hearntiow?\nKirki Pudoing toa?\nA: tin'